In [67]:
from pathlib import Path
import polars as pl
pl.Config.set_tbl_cols(-1)          # 显示全部列
pl.Config.set_tbl_width_chars(-1)   # 不限制表格总宽度

polars.config.Config

In [53]:
DATE = "20260624"
ROOT = Path("/data/xujiayi/xjy/l2")
EXCHANGE = "sz"
LEVELS = 10

# 浮点价格允许的误差
PRICE_TOLERANCE = 1e-8

# 数量理论上应完全一致；如有需要可改成1等其他值
QTY_TOLERANCE = 0.0

In [54]:
def null_safe_close(
    left: pl.Expr,
    right: pl.Expr,
    tolerance: float,
) -> pl.Expr:
    """两个值都为空，或者数值误差不超过 tolerance，即认为一致。"""
    return (
        (left.is_null() & right.is_null())
        | ((left - right).abs() <= tolerance)
    ).fill_null(False)


def to_float(column: str) -> pl.Expr:
    """兼容数字列以及包含双引号的字符串数字列。"""
    return (
        pl.col(column)
        .cast(pl.String)
        .str.strip_chars('"')
        .cast(pl.Float64, strict=False)
        .alias(column)
    )


In [55]:
# =============================================================================
# 1. 读取自己生成的快照
# =============================================================================

my_path = ROOT / "proc" / DATE / f"{EXCHANGE}shot_1m.pq"

my_shot = (
    pl.read_parquet(my_path)
    .with_columns(
        pl.col("SecurityID").cast(pl.Int64),
        pl.col("BarTime").cast(pl.Time),
    )
    .sort(["SecurityID", "BarTime"])
)

In [57]:
# =============================================================================
# 2. 读取官方快照
# =============================================================================

official_path = ROOT / "raw" / DATE / f"{EXCHANGE}shot.pq"

official_columns = ["UpdateTime", "SecurityID"]

for level in range(1, LEVELS + 1):
    official_columns.extend([
        f"BidPrice{level}",
        f"BidVolume{level}",
        f"AskPrice{level}",
        f"AskVolume{level}",
    ])

official_shot = (
    pl.read_parquet(official_path)
    .filter(pl.col("MDStreamID") == 10)
    .select(official_columns)
    .with_columns(
        pl.col("SecurityID").cast(pl.Int64),
        pl.col("UpdateTime").str.to_time(),
    )
    .with_columns([
        to_float(column)
        for column in official_columns
        if column not in {"UpdateTime", "SecurityID"}
    ])
    .sort(["SecurityID", "UpdateTime"])
)

In [58]:
# =============================================================================
# 3. 将官方快照对齐到自己生成的 BarTime
#
# 对每只证券、每个bar，选择：
#     UpdateTime <= BarTime
# 的最后一条官方快照。
# =============================================================================

bar_grid = (
    my_shot
    .select(["SecurityID", "BarTime"])
    .unique()
    .sort(["SecurityID", "BarTime"])
)

official_aligned = (
    bar_grid
    .join_asof(
        official_shot,
        left_on="BarTime",
        right_on="UpdateTime",
        by="SecurityID",
        strategy="backward",
        check_sortedness=False,
    )
    .with_columns(
        (
            (
                pl.col("BarTime").cast(pl.Int64)
                - pl.col("UpdateTime").cast(pl.Int64)
            )
            / 1_000_000_000
        ).alias("OfficialTimeGapSeconds")
    )
)

In [59]:
official_aligned.filter(pl.col('SecurityID') == 1).filter(
    (pl.col('BarTime')>=pl.time(9,30)) & (pl.col('BarTime')<=pl.time(15,0))
)

SecurityID,BarTime,UpdateTime,BidPrice1,BidVolume1,AskPrice1,AskVolume1,BidPrice2,BidVolume2,AskPrice2,AskVolume2,BidPrice3,BidVolume3,AskPrice3,AskVolume3,BidPrice4,BidVolume4,AskPrice4,AskVolume4,BidPrice5,BidVolume5,AskPrice5,AskVolume5,BidPrice6,BidVolume6,AskPrice6,AskVolume6,BidPrice7,BidVolume7,AskPrice7,AskVolume7,BidPrice8,BidVolume8,AskPrice8,AskVolume8,BidPrice9,BidVolume9,AskPrice9,AskVolume9,BidPrice10,BidVolume10,AskPrice10,AskVolume10,OfficialTimeGapSeconds
i64,time,time,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,09:30:00,09:30:00,10.67,1900.0,10.68,172700.0,10.66,149800.0,10.69,900.0,10.65,292100.0,10.7,2000.0,10.64,136200.0,10.71,98100.0,10.63,127300.0,10.72,45100.0,10.62,80900.0,10.73,120900.0,10.61,256900.0,10.74,5100.0,10.6,209000.0,10.75,81600.0,10.59,82300.0,10.76,41000.0,10.58,30000.0,10.77,74200.0,0.0
1,09:31:00,09:31:00,10.65,392900.0,10.66,32600.0,10.64,211800.0,10.67,79200.0,10.63,195300.0,10.68,82500.0,10.62,164000.0,10.69,57400.0,10.61,408400.0,10.7,7500.0,10.6,439400.0,10.71,63600.0,10.59,97600.0,10.72,50000.0,10.58,90300.0,10.73,130900.0,10.57,127300.0,10.74,7900.0,10.56,99400.0,10.75,88000.0,0.0
1,09:32:00,09:32:00,10.67,12000.0,10.68,1200.0,10.66,28600.0,10.69,23400.0,10.65,105600.0,10.7,65100.0,10.64,152500.0,10.71,40500.0,10.63,213700.0,10.72,52200.0,10.62,255500.0,10.73,136800.0,10.61,372800.0,10.74,14100.0,10.6,462300.0,10.75,91600.0,10.59,99500.0,10.76,61100.0,10.58,101300.0,10.77,83200.0,0.0
1,09:33:00,09:33:00,10.68,15200.0,10.69,53700.0,10.67,6300.0,10.7,67200.0,10.66,181900.0,10.71,25500.0,10.65,173700.0,10.72,52500.0,10.64,150700.0,10.73,137900.0,10.63,245300.0,10.74,18100.0,10.62,252400.0,10.75,101300.0,10.61,373100.0,10.76,63100.0,10.6,500200.0,10.77,85200.0,10.59,100500.0,10.78,230100.0,0.0
1,09:34:00,09:34:00,10.69,2300.0,10.7,85000.0,10.68,92500.0,10.71,52300.0,10.67,183700.0,10.72,54300.0,10.66,223600.0,10.73,139900.0,10.65,212400.0,10.74,28600.0,10.64,167200.0,10.75,111400.0,10.63,252700.0,10.76,66100.0,10.62,256000.0,10.77,85200.0,10.61,436400.0,10.78,232700.0,10.6,517200.0,10.79,207600.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,14:56:00,14:56:00,10.52,95600.0,10.53,29600.0,10.51,1.0419e6,10.54,160900.0,10.5,1.6335e6,10.55,254200.0,10.49,153900.0,10.56,60100.0,10.48,406600.0,10.57,154300.0,10.47,125500.0,10.58,131700.0,10.46,79300.0,10.59,111900.0,10.45,376100.0,10.6,119500.0,10.44,54100.0,10.61,56900.0,10.43,191800.0,10.62,121800.0,0.0
1,14:57:00,14:57:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
1,14:58:00,14:57:54,10.54,377823.0,10.54,377823.0,0.0,277.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6.0


In [60]:
my_shot.filter(pl.col('SecurityID') == 1).filter(
    (pl.col('BarTime')>=pl.time(9,30)) & (pl.col('BarTime')<=pl.time(15,0))
)

BarTime,SecurityID,BidPrice1,BidQty1,AskPrice1,AskQty1,BidPrice2,BidQty2,AskPrice2,AskQty2,BidPrice3,BidQty3,AskPrice3,AskQty3,BidPrice4,BidQty4,AskPrice4,AskQty4,BidPrice5,BidQty5,AskPrice5,AskQty5,BidPrice6,BidQty6,AskPrice6,AskQty6,BidPrice7,BidQty7,AskPrice7,AskQty7,BidPrice8,BidQty8,AskPrice8,AskQty8,BidPrice9,BidQty9,AskPrice9,AskQty9,BidPrice10,BidQty10,AskPrice10,AskQty10
time,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
09:30:00,1,10.68,41300.0,10.69,12300.0,10.67,87000.0,10.7,32500.0,10.66,72700.0,10.71,86100.0,10.65,247500.0,10.72,45000.0,10.64,84000.0,10.73,20200.0,10.63,122600.0,10.74,5100.0,10.62,76700.0,10.75,81600.0,10.61,238800.0,10.76,41000.0,10.6,191100.0,10.77,74200.0,10.59,79500.0,10.78,66500.0
09:31:00,1,10.65,392900.0,10.66,35800.0,10.64,211800.0,10.67,78400.0,10.63,195300.0,10.68,82500.0,10.62,164000.0,10.69,57400.0,10.61,408400.0,10.7,7500.0,10.6,439400.0,10.71,63600.0,10.59,97600.0,10.72,50000.0,10.58,90300.0,10.73,130900.0,10.57,127300.0,10.74,7900.0,10.56,99400.0,10.75,88000.0
09:32:00,1,10.67,12000.0,10.68,1200.0,10.66,28600.0,10.69,23400.0,10.65,105600.0,10.7,65100.0,10.64,152000.0,10.71,40500.0,10.63,213700.0,10.72,52200.0,10.62,256000.0,10.73,136800.0,10.61,372800.0,10.74,14100.0,10.6,462300.0,10.75,91600.0,10.59,99500.0,10.76,61100.0,10.58,101300.0,10.77,83200.0
09:33:00,1,10.68,15200.0,10.69,51500.0,10.67,5700.0,10.7,67200.0,10.66,178600.0,10.71,25500.0,10.65,173700.0,10.72,52500.0,10.64,150700.0,10.73,137900.0,10.63,245300.0,10.74,18100.0,10.62,252400.0,10.75,101300.0,10.61,373100.0,10.76,63100.0,10.6,500200.0,10.77,85200.0,10.59,100500.0,10.78,230100.0
09:34:00,1,10.69,2300.0,10.7,86600.0,10.68,92500.0,10.71,52300.0,10.67,183700.0,10.72,54300.0,10.66,223600.0,10.73,139900.0,10.65,212400.0,10.74,28600.0,10.64,167200.0,10.75,111400.0,10.63,252700.0,10.76,66100.0,10.62,256000.0,10.77,85200.0,10.61,436400.0,10.78,232700.0,10.6,517200.0,10.79,207600.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14:56:00,1,10.52,95600.0,10.53,29900.0,10.51,1.0419e6,10.54,157700.0,10.5,1.6335e6,10.55,254200.0,10.49,153900.0,10.56,60100.0,10.48,406600.0,10.57,154300.0,10.47,125500.0,10.58,131700.0,10.46,79300.0,10.59,111900.0,10.45,376100.0,10.6,119500.0,10.44,54100.0,10.61,56900.0,10.43,191800.0,10.62,121800.0
14:57:00,1,10.53,100.0,10.54,39900.0,10.52,111300.0,10.55,196300.0,10.51,1.0414e6,10.56,46100.0,10.5,1.5935e6,10.57,120400.0,10.49,138400.0,10.58,131700.0,10.48,405100.0,10.59,111800.0,10.47,119100.0,10.6,119500.0,10.46,79000.0,10.61,56900.0,10.45,375200.0,10.62,120800.0,10.44,49900.0,10.63,85000.0
14:58:00,1,11.78,3500.0,9.64,12000.0,11.06,12400.0,9.65,9600.0,11.0,100.0,10.23,900.0,10.86,25300.0,10.33,184923.0,10.85,15900.0,10.34,85200.0,10.75,266200.0,10.35,9600.0,10.57,3100.0,10.36,16000.0,10.56,5400.0,10.4,5000.0,10.55,10000.0,10.45,1000.0,10.54,49600.0,10.5,7600.0


In [61]:
# =============================================================================
# 4. 修改官方字段名，避免和自己的字段重名
# =============================================================================

official_rename = {}

for level in range(1, LEVELS + 1):
    official_rename.update({
        f"BidPrice{level}": f"OfficialBidPrice{level}",
        f"BidVolume{level}": f"OfficialBidQty{level}",
        f"AskPrice{level}": f"OfficialAskPrice{level}",
        f"AskVolume{level}": f"OfficialAskQty{level}",
    })

official_aligned = official_aligned.rename(official_rename)


# =============================================================================
# 5. 合并自己生成的快照和官方快照
# =============================================================================

comparison = my_shot.join(
    official_aligned,
    on=["SecurityID", "BarTime"],
    how="left",
)


In [ ]:
# =============================================================================
# 6. 逐档计算差值和是否一致
# =============================================================================

difference_expressions = []
match_expressions = []
match_columns = []

for level in range(1, LEVELS + 1):
    pairs = [
        (
            f"BidPrice{level}",
            f"OfficialBidPrice{level}",
            PRICE_TOLERANCE,
        ),
        (
            f"BidQty{level}",
            f"OfficialBidQty{level}",
            QTY_TOLERANCE,
        ),
        (
            f"AskPrice{level}",
            f"OfficialAskPrice{level}",
            PRICE_TOLERANCE,
        ),
        (
            f"AskQty{level}",
            f"OfficialAskQty{level}",
            QTY_TOLERANCE,
        ),
    ]

    for my_column, official_column, tolerance in pairs:
        diff_column = f"{my_column}Diff"
        match_column = f"{my_column}Match"

        difference_expressions.append(
            (
                pl.col(my_column)
                - pl.col(official_column)
            ).alias(diff_column)
        )

        match_expressions.append(
            null_safe_close(
                pl.col(my_column),
                pl.col(official_column),
                tolerance,
            ).alias(match_column)
        )

        match_columns.append(match_column)

comparison = (
    comparison
    .with_columns(difference_expressions)
    .with_columns(match_expressions)
    .with_columns(
        pl.all_horizontal(match_columns).alias("AllLevelsMatch")
    )
)

In [ ]:
# =============================================================================
# 7. 总体匹配率
# =============================================================================

summary_expressions = [
    pl.len().alias("RowCount"),
    pl.col("UpdateTime").is_not_null().sum().alias("OfficialMatchedRows"),
    pl.col("AllLevelsMatch").sum().alias("AllLevelsMatchedRows"),
    pl.col("AllLevelsMatch").mean().alias("AllLevelsMatchRate"),
]

for level in range(1, LEVELS + 1):
    for column in (
        f"BidPrice{level}",
        f"BidQty{level}",
        f"AskPrice{level}",
        f"AskQty{level}",
    ):
        summary_expressions.append(
            pl.col(f"{column}Match")
            .mean()
            .alias(f"{column}MatchRate")
        )

summary = comparison.select(summary_expressions)

print("总体匹配情况：")
print(summary)

In [64]:
# =============================================================================
# 8. 每一档的匹配率
# =============================================================================

level_summary_rows = []

for level in range(1, LEVELS + 1):
    result = comparison.select(
        pl.lit(level).alias("Level"),
        pl.col(f"BidPrice{level}Match").mean().alias("BidPriceMatchRate"),
        pl.col(f"BidQty{level}Match").mean().alias("BidQtyMatchRate"),
        pl.col(f"AskPrice{level}Match").mean().alias("AskPriceMatchRate"),
        pl.col(f"AskQty{level}Match").mean().alias("AskQtyMatchRate"),
    )

    level_summary_rows.append(result)

level_summary = pl.concat(level_summary_rows)

print("逐档匹配率：")
print(level_summary)

逐档匹配率：
shape: (10, 5)
┌───────┬───────────────────┬─────────────────┬───────────────────┬─────────────────┐
│ Level ┆ BidPriceMatchRate ┆ BidQtyMatchRate ┆ AskPriceMatchRate ┆ AskQtyMatchRate │
│ ---   ┆ ---               ┆ ---             ┆ ---               ┆ ---             │
│ i32   ┆ f64               ┆ f64             ┆ f64               ┆ f64             │
╞═══════╪═══════════════════╪═════════════════╪═══════════════════╪═════════════════╡
│ 1     ┆ 0.855936          ┆ 0.693188        ┆ 0.856996          ┆ 0.688568        │
│ 2     ┆ 0.84962           ┆ 0.79684         ┆ 0.851204          ┆ 0.795862        │
│ 3     ┆ 0.8479            ┆ 0.815896        ┆ 0.849526          ┆ 0.818204        │
│ 4     ┆ 0.846656          ┆ 0.829041        ┆ 0.848054          ┆ 0.831726        │
│ 5     ┆ 0.845897          ┆ 0.836541        ┆ 0.847216          ┆ 0.839068        │
│ 6     ┆ 0.845602          ┆ 0.840696        ┆ 0.846677          ┆ 0.843056        │
│ 7     ┆ 0.845561          ┆ 0.

In [65]:
# =============================================================================
# 9. 找出前两档不一致的记录
# =============================================================================

first_two_match_columns = []

for level in range(1, 3):
    first_two_match_columns.extend([
        f"BidPrice{level}Match",
        f"BidQty{level}Match",
        f"AskPrice{level}Match",
        f"AskQty{level}Match",
    ])

first_two_mismatches = (
    comparison
    .filter(
        ~pl.all_horizontal(first_two_match_columns)
    )
    .select([
        "BarTime",
        "UpdateTime",
        "OfficialTimeGapSeconds",
        "SecurityID",

        "BidPrice1",
        "OfficialBidPrice1",
        "BidPrice1Diff",
        "BidQty1",
        "OfficialBidQty1",
        "BidQty1Diff",

        "AskPrice1",
        "OfficialAskPrice1",
        "AskPrice1Diff",
        "AskQty1",
        "OfficialAskQty1",
        "AskQty1Diff",

        "BidPrice2",
        "OfficialBidPrice2",
        "BidPrice2Diff",
        "BidQty2",
        "OfficialBidQty2",
        "BidQty2Diff",

        "AskPrice2",
        "OfficialAskPrice2",
        "AskPrice2Diff",
        "AskQty2",
        "OfficialAskQty2",
        "AskQty2Diff",
    ])
    .sort(["SecurityID", "BarTime"])
)

print("前两档不一致记录：")
print(first_two_mismatches)

前两档不一致记录：
shape: (487_385, 28)
┌──────────┬────────────┬────────────┬───────────┬───┬───────────┬─────────┬───────────┬───────────┐
│ BarTime  ┆ UpdateTime ┆ OfficialTi ┆ SecurityI ┆ … ┆ AskPrice2 ┆ AskQty2 ┆ OfficialA ┆ AskQty2Di │
│ ---      ┆ ---        ┆ meGapSecon ┆ D         ┆   ┆ Diff      ┆ ---     ┆ skQty2    ┆ ff        │
│ time     ┆ time       ┆ ds         ┆ ---       ┆   ┆ ---       ┆ f64     ┆ ---       ┆ ---       │
│          ┆            ┆ ---        ┆ i64       ┆   ┆ f64       ┆         ┆ f64       ┆ f64       │
│          ┆            ┆ f64        ┆           ┆   ┆           ┆         ┆           ┆           │
╞══════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═════════╪═══════════╪═══════════╡
│ 09:16:00 ┆ 09:15:54   ┆ 6.0        ┆ 1         ┆ … ┆ 10.02     ┆ 100.0   ┆ 1800.0    ┆ -1700.0   │
│ 09:17:00 ┆ 09:16:57   ┆ 3.0        ┆ 1         ┆ … ┆ 10.02     ┆ 100.0   ┆ 3700.0    ┆ -3600.0   │
│ 09:18:00 ┆ 09:18:00   ┆ 0.0        ┆ 1         ┆ … ┆ null 

In [68]:
# =============================================================================
# 10. 单独检查某只证券、某个bar
# =============================================================================

sample = comparison.filter(
    (pl.col("SecurityID") == 1)
    & (pl.col("BarTime") == pl.time(14, 56))
).select([
    "SecurityID",
    "BarTime",
    "UpdateTime",
    "OfficialTimeGapSeconds",

    "BidPrice1",
    "OfficialBidPrice1",
    "BidQty1",
    "OfficialBidQty1",

    "AskPrice1",
    "OfficialAskPrice1",
    "AskQty1",
    "OfficialAskQty1",

    "BidPrice2",
    "OfficialBidPrice2",
    "BidQty2",
    "OfficialBidQty2",

    "AskPrice2",
    "OfficialAskPrice2",
    "AskQty2",
    "OfficialAskQty2",
])

print("单条样本：")
print(sample)

单条样本：
shape: (1, 20)
┌────────────┬──────────┬────────────┬────────────────────────┬───────────┬───────────────────┬─────────┬─────────────────┬───────────┬───────────────────┬─────────┬─────────────────┬───────────┬───────────────────┬──────────┬─────────────────┬───────────┬───────────────────┬──────────┬─────────────────┐
│ SecurityID ┆ BarTime  ┆ UpdateTime ┆ OfficialTimeGapSeconds ┆ BidPrice1 ┆ OfficialBidPrice1 ┆ BidQty1 ┆ OfficialBidQty1 ┆ AskPrice1 ┆ OfficialAskPrice1 ┆ AskQty1 ┆ OfficialAskQty1 ┆ BidPrice2 ┆ OfficialBidPrice2 ┆ BidQty2  ┆ OfficialBidQty2 ┆ AskPrice2 ┆ OfficialAskPrice2 ┆ AskQty2  ┆ OfficialAskQty2 │
│ ---        ┆ ---      ┆ ---        ┆ ---                    ┆ ---       ┆ ---               ┆ ---     ┆ ---             ┆ ---       ┆ ---               ┆ ---     ┆ ---             ┆ ---       ┆ ---               ┆ ---      ┆ ---             ┆ ---       ┆ ---               ┆ ---      ┆ ---             │
│ i64        ┆ time     ┆ time       ┆ f64                   